# Detection Via RT-DETR-L on Dairy Cow Dataset


## Step 0A: Runpod path configuration

In [ ]:
# RunPod path configuration
import os

os.environ["RUNPOD_NOTEBOOK_ROOT"] = "/workspace/**"
os.environ["DAIRY_COW_ROOT"] = "/workspace/**"
os.environ["SYNC_FROM_S3"] = ""
os.environ["FORCE_S3_SYNC"] = ""

print("RUNPOD_NOTEBOOK_ROOT:", os.environ["RUNPOD_NOTEBOOK_ROOT"])
print("DAIRY_COW_ROOT:", os.environ["**"])
print("SYNC_FROM_S3:", os.environ["SYNC_FROM_S3"])
print("FORCE_S3_SYNC:", os.environ["FORCE_S3_SYNC"])

## Step 0B: Verification of Packages

In [ ]:
import importlib
import shutil
import subprocess
import sys

REQUIRED_PACKAGES = {
    "yaml": "pyyaml",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "cv2": "opencv-python-headless",
    "PIL": "pillow",
    "psutil": "psutil",
    "tqdm": "tqdm",
    "ultralytics": "ultralytics",
    "torch": "torch",
}

missing = []
for module_name, package_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module_name)
    except Exception:
        missing.append(package_name)

print("Missing packages:", missing if missing else "None")

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    raise SystemExit("Installed missing Python packages. Restart the kernel, then rerun from Step 0.")

from ultralytics import RTDETR

aws_cli = shutil.which("aws")
print("AWS CLI found:", bool(aws_cli), aws_cli or "")

if aws_cli is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "awscli"])
    raise SystemExit("Installed awscli. Restart the kernel, then rerun from Step 0.")

print("Package, RT-DETR, and CLI verification passed.")

## Step 1: Libraries


In [ ]:
import gc
import json
import math
import os
import platform
import random
import re
import warnings
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import seaborn as sns
import torch
import yaml
from IPython.display import Image as IPyImage, display
from tqdm.auto import tqdm
from ultralytics import RTDETR

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

random.seed(42)
np.random.seed(42)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
print("Libraries imported successfully.")

## Step 2A: Dataset Pathways


In [ ]:
NOTEBOOK_DIR = Path(os.environ.get("RUNPOD_NOTEBOOK_ROOT", "/workspace/**")).resolve()
DATA_ROOT = Path(os.environ.get("DAIRY_COW_ROOT", "/workspace/**")).resolve()
RESULTS_ROOT = NOTEBOOK_DIR / "Results"

S3_BUCKET = os.environ.get("S3_BUCKET", "**")
S3_PREFIX = os.environ.get("S3_PREFIX", "**")
S3_DATASET_URI = f"s3://{S3_BUCKET}/{S3_PREFIX}/"
AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "**")

SYNC_FROM_S3 = os.environ.get("SYNC_FROM_S3", "").strip().lower() in {"", "*", "*", "*", "*"}
FORCE_S3_SYNC = os.environ.get("FORCE_S3_SYNC", "").strip().lower() in {"", "*", "*", "*", "*"}

EXPECTED_COUNTS = {
    "train": {"images": 171_284, "labels": 171_284},
    "val": {"images": 21_416, "labels": 21_416},
    "buffer": {"images": 21_487, "labels": 21_487},
}

NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("NOTEBOOK_DIR   :", NOTEBOOK_DIR)
print("DATA_ROOT      :", DATA_ROOT)
print("RESULTS_ROOT   :", RESULTS_ROOT)
print("S3_DATASET_URI :", S3_DATASET_URI)
print("AWS_REGION     :", AWS_REGION)
print("SYNC_FROM_S3   :", SYNC_FROM_S3)
print("FORCE_S3_SYNC  :", FORCE_S3_SYNC)
print("Dataset exists :", DATA_ROOT.exists())


## Step 2B: Syncing the dataset from Amazon S3


In [ ]:
# Step 2B: Optional S3 sync only if dataset is missing

import os
import subprocess

need_sync = FORCE_S3_SYNC or not DATA_ROOT.exists()

print("DATA_ROOT exists:", DATA_ROOT.exists())
print("FORCE_S3_SYNC   :", FORCE_S3_SYNC)
print("SYNC_FROM_S3    :", SYNC_FROM_S3)
print("Need S3 sync    :", need_sync)

if need_sync:
    if not SYNC_FROM_S3:
        raise RuntimeError(
            "Dataset is missing and SYNC_FROM_S3 is disabled. "
            "Either enable SYNC_FROM_S3=True or manually sync the dataset first."
        )

    # Check AWS CLI availability
    try:
        subprocess.check_call(["aws", "--version"])
    except Exception as exc:
        raise RuntimeError(
            "AWS CLI is not available in this RunPod environment. "
            "Install it first using: pip install awscli"
        ) from exc

    # Check whether AWS CLI can access the bucket.
    # This works whether credentials come from aws configure OR environment variables.
    test_cmd = [
        "aws", "s3", "ls",
        S3_DATASET_URI,
        "--region", AWS_REGION,
    ]

    print("Testing S3 access:")
    print(" ".join(test_cmd))

    try:
        subprocess.check_call(test_cmd)
    except Exception as exc:
        raise RuntimeError(
            "AWS S3 access failed. If this is a new RunPod pod, run this in the terminal first:\n\n"
            "aws configure\n\n"
            "Then enter privately:\n"
            "AWS Access Key ID\n"
            "AWS Secret Access Key\n"
            f"Default region name: {AWS_REGION}\n"
            "Default output format: json\n"
        ) from exc

    DATA_ROOT.mkdir(parents=True, exist_ok=True)

    sync_cmd = [
        "aws", "s3", "sync",
        S3_DATASET_URI,
        str(DATA_ROOT),
        "--region", AWS_REGION,
        "--only-show-errors",
    ]

    print("Running S3 sync:")
    print(" ".join(sync_cmd))

    subprocess.check_call(sync_cmd)

else:
    print("Skipping S3 sync because the dataset folder already exists.")

print("Dataset exists after Step 2B:", DATA_ROOT.exists())

## Step 2C: Dataset structure diagnostic

In [ ]:
EXPECTED_SPLIT_DIRS = [
    DATA_ROOT / "images" / "train",
    DATA_ROOT / "labels" / "train",
    DATA_ROOT / "images" / "val",
    DATA_ROOT / "labels" / "val",
    DATA_ROOT / "images" / "buffer",
    DATA_ROOT / "labels" / "buffer",
]

print("Dataset root:", DATA_ROOT)
print("Dataset root exists:", DATA_ROOT.exists())
print("Top-level contents:")
if DATA_ROOT.exists():
    for item in sorted(DATA_ROOT.iterdir()):
        kind = "dir" if item.is_dir() else "file"
        print(f" - [{kind}] {item.name}")
else:
    print(" - DATA_ROOT does not exist")

print("\nExpected split folders:")
missing_or_empty = []
for folder in EXPECTED_SPLIT_DIRS:
    if folder.exists():
        n_files = sum(1 for p in folder.iterdir() if p.is_file())
        print(f"{folder} | exists=True | files={n_files}")
        if n_files == 0:
            missing_or_empty.append(folder)
    else:
        print(f"{folder} | exists=False | files=NA")
        missing_or_empty.append(folder)

if missing_or_empty:
    print("\nDataset is incomplete at the configured DATA_ROOT.")
    print("First paths found under DATA_ROOT:")
    if DATA_ROOT.exists():
        for path in sorted(DATA_ROOT.rglob("*"))[:80]:
            print(" -", path)

    nested_candidates = []
    if DATA_ROOT.exists():
        for candidate in DATA_ROOT.rglob("images/train"):
            candidate_root = candidate.parent.parent
            if (candidate_root / "labels" / "train").exists():
                nested_candidates.append(candidate_root)

    if nested_candidates:
        print("\nPossible nested dataset roots:")
        for candidate in nested_candidates:
            print(" -", candidate)
        raise RuntimeError(
            "Dataset appears nested. Set DAIRY_COW_ROOT to the listed dataset root, rerun Step 2A, then continue."
        )

    raise RuntimeError(
        "Dataset folders are missing or empty. If the folder exists only as an incomplete placeholder, "
        "run os.environ['FORCE_S3_SYNC']='1', then rerun Steps 2A and 2B."
    )
print("Dataset structure diagnostic passed.")

## Step 2D: Inspecting a few image-label pairs


In [ ]:
def label_path_for_image(img_path: Path, label_dir: Path) -> Path:
    return label_dir / f"{img_path.stem}.txt"

TRAIN_IMG_DIR = DATA_ROOT / "images" / "train"
TRAIN_LBL_DIR = DATA_ROOT / "labels" / "train"

train_images = sorted([p for p in TRAIN_IMG_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
train_labels = sorted(TRAIN_LBL_DIR.glob("*.txt"))

print("Training images:", len(train_images))
print("Training labels:", len(train_labels))

for img_path in train_images[:5]:
    lbl_path = label_path_for_image(img_path, TRAIN_LBL_DIR)
    print("\nImage:", img_path.name)
    print("Label:", lbl_path.name, "| Exists:", lbl_path.exists())
    if lbl_path.exists():
        with open(lbl_path, "r", encoding="utf-8") as f:
            for line in f.readlines()[:5]:
                print(" ", line.strip())


## Step 3A: Machine/Laptop Specifications


In [ ]:
print("System Information")
print("OS:", platform.system(), platform.release())
print("Python version:", platform.python_version())
print("Processor:", platform.processor())
print("Machine:", platform.machine())

print("\nCPU/RAM")
print("Physical cores:", psutil.cpu_count(logical=False))
print("Logical cores:", psutil.cpu_count(logical=True))
print("Total RAM (GB):", round(psutil.virtual_memory().total / (1024**3), 2))
print("Available RAM:", round(psutil.virtual_memory().available / (1024**3), 2))

print("\nStorage")
disk_root = "/workspace" if Path("/workspace").exists() else str(NOTEBOOK_DIR.anchor or NOTEBOOK_DIR)
disk = psutil.disk_usage(disk_root)
print("Total disk (GB):", round(disk.total / (1024**3), 2))
print("Used disk (GB):", round(disk.used / (1024**3), 2))
print("Free disk (GB):", round(disk.free / (1024**3), 2))

print("\nPYTORCH / CUDA")
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU device:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("GPU memory (GB):", round(props.total_memory / (1024**3), 2))
else:
    print("GPU device: CPU only")


## Step 3B: Settings for GPU


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not detected. Do not continue on CPU.")

gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
cpu_workers = os.cpu_count() or 4

recommended_model = "rtdetr-l.pt"
recommended_imgsz = 640
recommended_epochs = 40
recommended_patience = 0
recommended_device = 0

if gpu_mem_gb >= 80:
    recommended_batch = 16
elif gpu_mem_gb >= 48:
    recommended_batch = 8
elif gpu_mem_gb >= 24:
    recommended_batch = 4
else:
    raise RuntimeError(
        f"Only {gpu_mem_gb:.2f} GB VRAM detected. Use a 24 GB+ GPU for RT-DETR-L, preferably 48 GB+."
    )

recommended_workers = max(2, min(4, cpu_workers // 2))

print("Recommended model   :", recommended_model)
print("Recommended imgsz   :", recommended_imgsz)
print("Recommended batch   :", recommended_batch)
print("Recommended workers :", recommended_workers)
print("Recommended epochs  :", recommended_epochs)
print("Recommended patience:", recommended_patience)
print("Recommended device  :", recommended_device)

## Step 4A: Dairy Cow dataset verification check


In [ ]:
def count_files(folder: Path, suffixes=None) -> int:
    if not folder.exists():
        return 0
    if suffixes is None:
        return sum(1 for p in folder.iterdir() if p.is_file())
    return sum(1 for p in folder.iterdir() if p.is_file() and p.suffix.lower() in suffixes)

required_dirs = [
    DATA_ROOT / "images" / "train",
    DATA_ROOT / "images" / "val",
    DATA_ROOT / "images" / "buffer",
    DATA_ROOT / "labels" / "train",
    DATA_ROOT / "labels" / "val",
    DATA_ROOT / "labels" / "buffer",
]

missing_dirs = [str(p) for p in required_dirs if not p.exists()]
print("Checking dataset root:", DATA_ROOT)
if missing_dirs:
    print("\nMissing required folders:")
    for p in missing_dirs:
        print(" -", p)
    raise FileNotFoundError("Dataset structure is incomplete.")

split_counts = []
for split_name in ["train", "val", "buffer"]:
    img_dir = DATA_ROOT / "images" / split_name
    lbl_dir = DATA_ROOT / "labels" / split_name
    n_images = count_files(img_dir, IMAGE_EXTS)
    n_labels = count_files(lbl_dir, {".txt"})
    split_counts.append({
        "split": split_name,
        "images": n_images,
        "labels": n_labels,
        "difference": n_images - n_labels,
        "expected_images": EXPECTED_COUNTS[split_name]["images"],
        "expected_labels": EXPECTED_COUNTS[split_name]["labels"],
    })

df_split_counts = pd.DataFrame(split_counts)
display(df_split_counts)

if not (df_split_counts["images"] > 0).all():
    raise RuntimeError("At least one image split is empty.")

if not (df_split_counts["difference"] == 0).all():
    raise RuntimeError("Image-label count mismatch detected.")

print("Dataset verification passed.")

## Step 4B: Inspecting the dataset structure


In [ ]:
SOURCE_YAML_CANDIDATES = [
    DATA_ROOT / "data_full_detection.yaml",
    DATA_ROOT / "data.yaml",
]

SOURCE_YAML = next((p for p in SOURCE_YAML_CANDIDATES if p.exists()), None)
if SOURCE_YAML is None:
    raise FileNotFoundError("No source YAML file was found in the dataset directory.")

with open(SOURCE_YAML, "r", encoding="utf-8") as f:
    source_cfg = yaml.safe_load(f)

names = source_cfg.get("names", ["Drinking", "Feeding", "Resting", "Standing", "Walking"])
if isinstance(names, dict):
    class_map = {int(k): v for k, v in names.items()}
    names_list = [class_map[i] for i in sorted(class_map)]
else:
    names_list = list(names)
    class_map = {i: name for i, name in enumerate(names_list)}

DATA_YAML_PATH = NOTEBOOK_DIR / "data_dc_det_rtdetr_l_runpod.yaml"
data_cfg = {
    "path": str(DATA_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/buffer",
    "nc": len(names_list),
    "names": names_list,
}
with open(DATA_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

print("Dataset directory :", DATA_ROOT.resolve())
print("Source YAML       :", SOURCE_YAML.resolve())
print("Detection YAML    :", DATA_YAML_PATH.resolve())
print(yaml.safe_dump(data_cfg, sort_keys=False))

## Step 4C: Dairy Cow dataset inspection and data structure


In [ ]:
def get_light_condition(filename):
    name = filename.lower()
    if "night" in name:
        return "night"
    elif "day" in name:
        return "day"
    else:
        return "unknown"

def extract_date_from_filename(filename):
    match = re.search(r"(20\d{6})", filename)
    if match:
        return match.group(1)
    return "unknown"

def extract_hour_from_filename(filename):
    match = re.search(r"(20\d{6})(\d{2})(\d{2})(\d{2})", filename)
    if match:
        return match.group(2)
    return "unknown"

def read_label_classes(label_path: Path):
    cls_ids = []
    if not label_path.exists():
        return cls_ids
    with open(label_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                try:
                    cls_ids.append(int(float(parts[0])))
                except ValueError:
                    pass
    return cls_ids

rows = []
split_map = {"train": "train", "val": "val", "test": "buffer"}
for report_split, folder_split in split_map.items():
    img_dir = DATA_ROOT / "images" / folder_split
    lbl_dir = DATA_ROOT / "labels" / folder_split

    for img_path in sorted(img_dir.iterdir()):
        if img_path.suffix.lower() not in IMAGE_EXTS:
            continue

        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        cls_ids = read_label_classes(lbl_path)

        row = {
            "split": report_split,
            "filename": img_path.name,
            "light_condition": get_light_condition(img_path.name),
            "date": extract_date_from_filename(img_path.name),
            "hour": extract_hour_from_filename(img_path.name),
            "num_instances": len(cls_ids),
            "classes_in_image": sorted(set(cls_ids)),
        }

        for class_id, class_name in class_map.items():
            row[f"has_{class_name}"] = int(class_id in cls_ids)
            row[f"count_{class_name}"] = cls_ids.count(class_id)

        rows.append(row)

df_audit = pd.DataFrame(rows)
print("Total images inspected:", len(df_audit))
display(df_audit.head())

split_table = df_audit["split"].value_counts().rename_axis("split").reset_index(name="images")
light_table = df_audit["light_condition"].value_counts().rename_axis("light_condition").reset_index(name="images")
date_table = df_audit["date"].value_counts().sort_index().rename_axis("date").reset_index(name="images")
hour_table = df_audit["hour"].value_counts().sort_index().rename_axis("hour").reset_index(name="images")

print("\n1) Split distribution")
display(split_table)

print("\n2) Day and Nighttime distribution")
display(light_table)

print("\n3) Dairy Cow dataset distribution")
display(date_table)

print("\n4) Dairy Cow hourly distribution")
display(hour_table)

split_light = pd.crosstab(df_audit["split"], df_audit["light_condition"])
split_date = pd.crosstab(df_audit["split"], df_audit["date"])
split_hour = pd.crosstab(df_audit["split"], df_audit["hour"])

print("\n5) Split vs. Light condition")
display(split_light)

print("\n6) Split vs. Date")
display(split_date)

print("\n7) Split vs. Hour")
display(split_hour)

image_presence_rows = []
for class_id, class_name in class_map.items():
    col = f"has_{class_name}"
    total_images = int(df_audit[col].sum())
    image_presence_rows.append({
        "class_id": class_id,
        "class_name": class_name,
        "images_containing_class": total_images
    })
df_image_presence = pd.DataFrame(image_presence_rows).sort_values(
    "images_containing_class", ascending=False
)
print("\n8) Class distribution by image presence")
display(df_image_presence)

instance_rows = []
for class_id, class_name in class_map.items():
    col = f"count_{class_name}"
    total_instances = int(df_audit[col].sum())
    instance_rows.append({
        "class_id": class_id,
        "class_name": class_name,
        "total_instances": total_instances
    })
df_instances = pd.DataFrame(instance_rows).sort_values(
    "total_instances", ascending=False
)
print("\n9) Class distribution by total number of object instances")
display(df_instances)

split_class_rows = []
for split in ["train", "val", "test"]:
    df_split = df_audit[df_audit["split"] == split]
    row = {"split": split}
    for class_id, class_name in class_map.items():
        row[class_name] = int(df_split[f"count_{class_name}"].sum())
    split_class_rows.append(row)
df_split_class_instances = pd.DataFrame(split_class_rows)
print("\n10) Split vs. class instance counts")
display(df_split_class_instances)

cache_files = list(DATA_ROOT.rglob("*.cache"))
print("\nCache files found:", len(cache_files))
for p in cache_files:
    try:
        p.unlink()
    except Exception as e:
        print("Could not delete:", p, "|", e)
print("Cache cleanup completed.")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

plt.figure()
sns.barplot(data=split_table, x="split", y="images")
plt.title("Dairy Cow Dataset Split Distribution")
plt.xlabel("Split")
plt.ylabel("Number of Images")
plt.show()

plt.figure()
sns.barplot(data=light_table, x="light_condition", y="images")
plt.title("Day vs. Night Distribution")
plt.xlabel("Light condition")
plt.ylabel("Number of Images")
plt.show()

plt.figure(figsize=(12, 6))
sns.barplot(data=hour_table, x="hour", y="images")
plt.title("Hourly Images Distribution")
plt.xlabel("Hours")
plt.ylabel("Number of Images")
plt.show()

plt.figure()
sns.barplot(data=df_image_presence, x="class_name", y="images_containing_class")
plt.title("Images Containing Each Behavior")
plt.xlabel("Behavior Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=20)
plt.show()

plt.figure()
sns.barplot(data=df_instances, x="class_name", y="total_instances")
plt.title("Total Object Instances per Behavior")
plt.xlabel("Behavior Class")
plt.ylabel("Number of Instances")
plt.xticks(rotation=20)
plt.show()

## Step 5: Dairy Cow dataset pathway and RT-DETR-L settings


In [ ]:
RUN_NAME = os.environ.get("RTDETR_RUN_NAME", "rtdetr_l_dairy_cow_full_runpod")
MODEL_NAME = os.environ.get("RTDETR_MODEL", recommended_model)

EPOCHS = int(os.environ.get("RTDETR_EPOCHS", recommended_epochs))
IMGSZ = int(os.environ.get("RTDETR_IMGSZ", recommended_imgsz))
BATCH = int(os.environ.get("RTDETR_BATCH", recommended_batch))
WORKERS = int(os.environ.get("RTDETR_WORKERS", recommended_workers))
PATIENCE = int(os.environ.get("RTDETR_PATIENCE", recommended_patience))
DEVICE = os.environ.get("RTDETR_DEVICE", str(recommended_device))
PROJECT_DIR = RESULTS_ROOT / "Detection" / "RT-DETR"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

if isinstance(DEVICE, str) and DEVICE.isdigit():
    DEVICE = int(DEVICE)

if MODEL_NAME == "rtdetr-x.pt" and gpu_mem_gb < 80:
    raise RuntimeError(
        f"RT-DETR-X was requested on only {gpu_mem_gb:.2f} GB VRAM. Use an 80 GB+ GPU or switch to rtdetr-l.pt."
    )

print("Model    :", MODEL_NAME)
print("Data     :", DATA_YAML_PATH)
print("Project  :", PROJECT_DIR)
print("Run name :", RUN_NAME)
print("Device   :", DEVICE)
print("Epochs   :", EPOCHS)
print("Img size :", IMGSZ)
print("Batch    :", BATCH)
print("Workers  :", WORKERS)
print("Patience :", PATIENCE)

## Step 6: GPU stability settings


In [ ]:
import gc
import shutil
import torch
from pathlib import Path

PROJECT_DIR = RESULTS_ROOT / "Detection" / "RT-DETR"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

RUN_NAME = os.environ.get("RTDETR_RUN_NAME", RUN_NAME if "RUN_NAME" in globals() else "rtdetr_l_dairy_cow_full_runpod")
RUN_DIR = PROJECT_DIR / RUN_NAME

EVAL_DIR = RUN_DIR / "ev"
FIG_DIR = RUN_DIR / "fig"

old_files = [
    RUN_DIR / "met.csv",
    RUN_DIR / "met.json",
    RUN_DIR / "pc.csv",
    RUN_DIR / "cmp.csv",
    RUN_DIR / "cmp.xlsx",
    RUN_DIR / "sum.json",
]

print("Project dir :", PROJECT_DIR)
print("Run dir     :", RUN_DIR)

if RUN_DIR.exists():
    print("Removing old run directory:", RUN_DIR)
    shutil.rmtree(RUN_DIR)

for cache_file in DATA_ROOT.rglob("*.cache"):
    try:
        cache_file.unlink()
        print("Deleted cache:", cache_file)
    except Exception as exc:
        print("Could not delete cache:", cache_file, exc)

for old_file in old_files:
    if old_file.exists():
        try:
            old_file.unlink()
            print("Deleted old file:", old_file)
        except Exception as exc:
            print("Could not delete old file:", old_file, exc)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("GPU stability settings applied.")
if torch.cuda.is_available():
    print("GPU name      :", torch.cuda.get_device_name(0))
    print("GPU memory GB :", round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2))
print("CUDA benchmark:", torch.backends.cudnn.benchmark)
print("TF32 matmul   :", torch.backends.cuda.matmul.allow_tf32 if torch.cuda.is_available() else "N/A")
print("TF32 cuDNN    :", torch.backends.cudnn.allow_tf32 if torch.cuda.is_available() else "N/A")

## Step 7: Dairy Cow dataset training on RT-DETR-L


In [ ]:
import gc
import time
import torch
from pathlib import Path
from tqdm.auto import tqdm
from ultralytics import RTDETR

MODEL_NAME = os.environ.get("RTDETR_MODEL", MODEL_NAME if "MODEL_NAME" in globals() else "rtdetr-l.pt")
EPOCHS = int(os.environ.get("RTDETR_EPOCHS", "40"))
IMGSZ = int(os.environ.get("RTDETR_IMGSZ", "640"))
BATCH = int(os.environ.get("RTDETR_BATCH", str(BATCH if "BATCH" in globals() else 8)))
WORKERS = int(os.environ.get("RTDETR_WORKERS", str(WORKERS if "WORKERS" in globals() else 4)))
PATIENCE = int(os.environ.get("RTDETR_PATIENCE", "0"))
DEVICE = int(os.environ.get("RTDETR_DEVICE", "0"))

DATA_YAML_PATH = Path(DATA_YAML_PATH)
DATA_ROOT = Path(DATA_ROOT)
PROJECT_DIR = RESULTS_ROOT / "Detection" / "RT-DETR"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = PROJECT_DIR / RUN_NAME

torch.cuda.empty_cache()
gc.collect()

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("Training setup")
print("Model     :", MODEL_NAME)
print("Data YAML :", DATA_YAML_PATH)
print("Data root :", DATA_ROOT)
print("Run name  :", RUN_NAME)
print("Run dir   :", RUN_DIR)
print("Epochs    :", EPOCHS)
print("Img size  :", IMGSZ)
print("Batch     :", BATCH)
print("Workers   :", WORKERS)
print("Patience  :", PATIENCE)
print("Device    :", DEVICE)
print("Cache     : False")
print("Plots     : True")
print("Save per  : 5")

progress = {"bar": None, "start": None}

def on_train_epoch_start(trainer):
    total_batches = len(trainer.train_loader)
    progress["start"] = time.time()
    if progress["bar"] is not None:
        progress["bar"].close()
    progress["bar"] = tqdm(
        total=total_batches,
        desc=f"Epoch {trainer.epoch + 1}/{trainer.epochs}",
        unit="batch",
        dynamic_ncols=True,
        ascii=True,
        leave=True,
    )

def on_train_batch_end(trainer):
    bar = progress.get("bar")
    if bar is None:
        return
    bar.update(1)
    postfix = {}
    try:
        if torch.cuda.is_available():
            postfix["gpu_gb"] = f"{torch.cuda.memory_reserved(0) / (1024 ** 3):.2f}"
    except Exception:
        pass
    try:
        items = getattr(trainer, "loss_items", None)
        if items is not None and len(items) >= 3:
            postfix["loss_0"] = f"{float(items[0]):.4f}"
            postfix["loss_1"] = f"{float(items[1]):.4f}"
            postfix["loss_2"] = f"{float(items[2]):.4f}"
    except Exception:
        pass
    try:
        elapsed = max(time.time() - progress["start"], 1e-6)
        speed = bar.n / elapsed
        remain = (bar.total - bar.n) / max(speed, 1e-6)
        postfix["it_s"] = f"{speed:.2f}"
        postfix["eta_m"] = f"{remain / 60:.1f}"
    except Exception:
        pass
    if postfix:
        bar.set_postfix(postfix)

def on_train_epoch_end(trainer):
    bar = progress.get("bar")
    if bar is not None:
        bar.close()
        progress["bar"] = None
    if progress["start"] is not None:
        print(f"Epoch {trainer.epoch + 1} completed in {(time.time() - progress['start']) / 60:.2f} minutes")

def on_train_end(trainer):
    bar = progress.get("bar")
    if bar is not None:
        bar.close()
        progress["bar"] = None
    print("Training finished.")
    print("Save dir:", trainer.save_dir)

model = RTDETR(MODEL_NAME)
model.add_callback("on_train_epoch_start", on_train_epoch_start)
model.add_callback("on_train_batch_end", on_train_batch_end)
model.add_callback("on_train_epoch_end", on_train_epoch_end)
model.add_callback("on_train_end", on_train_end)

train_results = model.train(
    data=str(DATA_YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    patience=PATIENCE,
    device=DEVICE,
    pretrained=True,
    optimizer="auto",
    cache=False,
    amp=True,
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    verbose=False,
    save=True,
    save_period=5,
    val=True,
)

RUN_DIR = Path(train_results.save_dir)
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"

print("Training completed.")
print("Run dir :", RUN_DIR)
print("Best pt :", BEST_PT)
print("Last pt :", LAST_PT)

## Step 8: Evaluation on train, validation, and test


In [ ]:
import json
import numpy as np
import pandas as pd
from ultralytics import RTDETR

BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"

print("Run dir :", RUN_DIR)
print("Best pt :", BEST_PT.exists())
print("Last pt :", LAST_PT.exists())

assert BEST_PT.exists(), f"best.pt not found at: {BEST_PT}"

best_model = RTDETR(str(BEST_PT))

EVAL_DIR = RUN_DIR / "ev"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

EVAL_BATCH = min(BATCH, 24)
EVAL_WORKERS = 2

split_metrics = {}
split_dirs = {"train": "tr", "val": "va", "test": "te"}

for split_name, short_name in split_dirs.items():
    print(f"\nEvaluating {split_name} split")
    metrics_obj = best_model.val(
        data=str(DATA_YAML_PATH),
        split=split_name,
        imgsz=IMGSZ,
        batch=EVAL_BATCH,
        device=DEVICE,
        workers=EVAL_WORKERS,
        plots=True,
        save_json=True,
        project=str(EVAL_DIR),
        name=short_name,
        exist_ok=True,
        verbose=True,
    )

    precision = float(metrics_obj.results_dict.get("metrics/precision(B)", np.nan))
    recall = float(metrics_obj.results_dict.get("metrics/recall(B)", np.nan))
    map50 = float(metrics_obj.results_dict.get("metrics/mAP50(B)", metrics_obj.box.map50))
    map5095 = float(metrics_obj.results_dict.get("metrics/mAP50-95(B)", metrics_obj.box.map))
    map75 = float(metrics_obj.box.map75)
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else np.nan

    split_metrics[split_name] = {
        "precision": precision,
        "recall": recall,
        "f1": float(f1) if not np.isnan(f1) else np.nan,
        "mAP50": map50,
        "mAP75": map75,
        "mAP50_95": map5095,
        "pcm": [float(x) for x in metrics_obj.box.maps],
        "dir": str(EVAL_DIR / short_name),
    }

    print(f"{split_name} precision : {precision:.4f}")
    print(f"{split_name} recall    : {recall:.4f}")
    print(f"{split_name} F1-score  : {f1:.4f}")
    print(f"{split_name} mAP50     : {map50:.4f}")
    print(f"{split_name} mAP75     : {map75:.4f}")
    print(f"{split_name} mAP50-95  : {map5095:.4f}")

df_split_metrics = pd.DataFrame(split_metrics).T.reset_index().rename(columns={"index": "split"})

MET_CSV = RUN_DIR / "met.csv"
MET_JSON = RUN_DIR / "met.json"
df_split_metrics.to_csv(MET_CSV, index=False)
with open(MET_JSON, "w", encoding="utf-8") as f:
    json.dump(split_metrics, f, indent=4)

print("\nSaved:", MET_CSV)
print("Saved:", MET_JSON)
display(df_split_metrics)

## Step 9: Visualization


In [ ]:
from IPython.display import Image as IPyImage, display

train_plot_names = [
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxPR_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
    "BoxF1_curve.png",
    "labels.jpg",
]

print("Training plots")
for plot_name in train_plot_names:
    plot_path = RUN_DIR / plot_name
    if plot_path.exists():
        print("Displaying:", plot_name)
        display(IPyImage(filename=str(plot_path), width=900))
    else:
        print("Missing:", plot_path)

eval_plot_names = [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxPR_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
    "BoxF1_curve.png",
]

for split_name, short_name in split_dirs.items():
    split_dir = EVAL_DIR / short_name
    print(f"\nEvaluation plots for {split_name}")
    print("Folder:", split_dir)
    for plot_name in eval_plot_names:
        plot_path = split_dir / plot_name
        if plot_path.exists():
            print("Displaying:", plot_name)
            display(IPyImage(filename=str(plot_path), width=900))
        else:
            print("Missing:", plot_path)

## Step 10: Inspection of results.csv and visualization of training curves


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RES_CSV = RUN_DIR / "results.csv"
assert RES_CSV.exists(), f"results.csv not found at: {RES_CSV}"

df_results = pd.read_csv(RES_CSV)
df_results.columns = [c.strip() for c in df_results.columns]

print("Loaded:", RES_CSV)
display(df_results.head())

FIG_DIR = RUN_DIR / "fig"
FIG_DIR.mkdir(parents=True, exist_ok=True)

if "epoch" in df_results.columns:
    x_epochs = df_results["epoch"] + 1
else:
    x_epochs = np.arange(1, len(df_results) + 1)

def save_curve(y_col, title, y_label, file_name):
    if y_col not in df_results.columns:
        print("Missing column:", y_col)
        return

    fig, ax = plt.subplots(figsize=(7.5, 5.2))
    ax.plot(x_epochs, df_results[y_col], marker="o", linewidth=2, markersize=4)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Epoch", fontsize=12, fontweight="bold")
    ax.set_ylabel(y_label, fontsize=12, fontweight="bold")
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
    ax.tick_params(axis="both", labelsize=11)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    fig.tight_layout()

    png_path = FIG_DIR / f"{file_name}.png"
    pdf_path = FIG_DIR / f"{file_name}.pdf"
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.show()
    print("Saved:", png_path)
    print("Saved:", pdf_path)

curve_specs = [
    ("train/box_loss", "Training Box Loss", "Box loss", "tr_box"),
    ("train/cls_loss", "Training Classification Loss", "Classification loss", "tr_cls"),
    ("train/dfl_loss", "Training Distribution Focal Loss", "DFL loss", "tr_dfl"),
    ("val/box_loss", "Validation Box Loss", "Box loss", "va_box"),
    ("val/cls_loss", "Validation Classification Loss", "Classification loss", "va_cls"),
    ("val/dfl_loss", "Validation Distribution Focal Loss", "DFL loss", "va_dfl"),
    ("metrics/precision(B)", "Validation Precision", "Precision", "va_p"),
    ("metrics/recall(B)", "Validation Recall", "Recall", "va_r"),
    ("metrics/mAP50(B)", "Validation mAP50", "mAP50", "va_m50"),
    ("metrics/mAP50-95(B)", "Validation mAP50-95", "mAP50-95", "va_m95"),
]

for y_col, title, y_label, file_name in curve_specs:
    save_curve(y_col, title, y_label, file_name)

metric_col = "metrics/mAP50-95(B)"
if metric_col in df_results.columns:
    best_epoch = int(df_results[metric_col].idxmax()) + 1
    best_score = float(df_results[metric_col].max())
    print("\nBest epoch based on validation mAP50-95:", best_epoch)
    print("Best validation mAP50-95:", round(best_score, 4))

## Step 11: Saving RT-DETR experiment summary


In [ ]:
import json
import numpy as np
import pandas as pd

names = best_model.names
class_names = [names[i] for i in sorted(names.keys())]

per_class_rows = []
for split_name in ["train", "val", "test"]:
    per_class_maps = split_metrics[split_name]["pcm"]
    for class_index, class_name in enumerate(class_names):
        value = per_class_maps[class_index] if class_index < len(per_class_maps) else np.nan
        per_class_rows.append({
            "split": split_name,
            "class_index": class_index,
            "class_name": class_name,
            "mAP50_95": value,
        })

df_per_class = pd.DataFrame(per_class_rows)
PC_CSV = RUN_DIR / "pc.csv"
df_per_class.to_csv(PC_CSV, index=False)
print("Saved:", PC_CSV)
display(df_per_class)

summary = {
    "run_name": RUN_DIR.name,
    "model": MODEL_NAME,
    "epochs": EPOCHS,
    "image_size": IMGSZ,
    "batch_size": BATCH,
    "workers": WORKERS,
    "patience": PATIENCE,
    "device": str(DEVICE),
    "dataset_root": str(DATA_ROOT),
    "data_yaml": str(DATA_YAML_PATH),
    "best_weights": str(BEST_PT),
    "last_weights": str(LAST_PT) if LAST_PT.exists() else None,
    "results_csv": str(RES_CSV),
    "metrics_csv": str(MET_CSV),
    "metrics_json": str(MET_JSON),
    "per_class_csv": str(PC_CSV),
    "fig_dir": str(FIG_DIR),
    "eval_dir": str(EVAL_DIR),
    "split_metrics": split_metrics,
}

if "df_split_counts" in globals():
    summary["split_inventory"] = df_split_counts.to_dict(orient="records")
if "S3_DATASET_URI" in globals():
    summary["s3_dataset_uri"] = S3_DATASET_URI
if "metric_col" in globals() and metric_col in df_results.columns:
    summary["best_epoch"] = best_epoch
    summary["best_val_mAP50_95"] = best_score

SUM_JSON = RUN_DIR / "sum.json"
with open(SUM_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

print("Saved:", SUM_JSON)
print(json.dumps(summary, indent=4))

## Step 12: Comparison Table


In [ ]:
import sys
import subprocess
import pandas as pd

comparison_row = pd.DataFrame([{
    "model": MODEL_NAME,
    "run_name": RUN_DIR.name,
    "epochs": EPOCHS,
    "image_size": IMGSZ,
    "batch_size": BATCH,

    "train_precision": split_metrics["train"]["precision"],
    "train_recall": split_metrics["train"]["recall"],
    "train_f1": split_metrics["train"]["f1"],
    "train_mAP50": split_metrics["train"]["mAP50"],
    "train_mAP75": split_metrics["train"]["mAP75"],
    "train_mAP50_95": split_metrics["train"]["mAP50_95"],

    "val_precision": split_metrics["val"]["precision"],
    "val_recall": split_metrics["val"]["recall"],
    "val_f1": split_metrics["val"]["f1"],
    "val_mAP50": split_metrics["val"]["mAP50"],
    "val_mAP75": split_metrics["val"]["mAP75"],
    "val_mAP50_95": split_metrics["val"]["mAP50_95"],

    "test_precision": split_metrics["test"]["precision"],
    "test_recall": split_metrics["test"]["recall"],
    "test_f1": split_metrics["test"]["f1"],
    "test_mAP50": split_metrics["test"]["mAP50"],
    "test_mAP75": split_metrics["test"]["mAP75"],
    "test_mAP50_95": split_metrics["test"]["mAP50_95"],
}])

CMP_CSV = RUN_DIR / "cmp.csv"
CMP_XLSX = RUN_DIR / "cmp.xlsx"
comparison_row.to_csv(CMP_CSV, index=False)
print("Saved CSV:", CMP_CSV)

try:
    import openpyxl
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])

comparison_row.to_excel(CMP_XLSX, index=False, engine="openpyxl")
print("Saved Excel:", CMP_XLSX)

display(comparison_row)